<a href="https://colab.research.google.com/github/DuaaMahar5/FlyRank-internship-ml/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DuaaMahar5/FlyRank-internship-ml/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

Ranking/scoring. The content team needs a prioritized queue across ~30k pages, not a single yes/no per page. I'll likely train on a label and use its output score to rank.

In [34]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

Target = is_declining_label, defined as whether a page's engagement trend (last 30 days vs. prior 30 days) is downward. This is an observed trend, not an arbitrary rule — it's computed directly from measured traffic windows.

 However, it's a proxy for the real question ("will refreshing this page help?"), since the dataset has no before/after record of refresh outcomes. I'll treat model output as prioritization support, not a guarantee that refreshing declining pages will reverse the trend.

In [35]:
print(lane_slice['is_declining_label'].value_counts())

is_declining_label
1    16262
0    13738
Name: count, dtype: int64


## 3. Success metric

*One metric you can defend. What number means 'good'?*

Precision@50 — of the top 50 pages ranked as declining, what fraction actually are. This matches how the content team will use the output (working top-down through a queue, not reviewing all 30k pages).


 The starter repo's reference pipeline already shows this is computable on this data (baseline rule ≈0.24 → trained model ≈0.74), which tells me the metric is measurable here — I still need to confirm it holds once I build my own version

In [36]:
# Check the overall proportion of declining pages, which informs the baseline for Precision@50.
print(f"Proportion of declining pages: {lane_slice['is_declining_label'].mean():.2f}")

Proportion of declining pages: 0.54


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

One row = one content page (content_id) eligible for refresh scoring — meaning it has real search impressions and is at least 90 days old (new pages haven't had time to show a trend yet).

 Each row is one client's page, aggregated over a 90-day window, carrying its traffic, engagement, and trend signals

In [37]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# my lane's slice: pages with real traffic, old enough to show a trend
lane_slice = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].copy()

# Create the 'is_declining_label' from 'trend_direction' as per the skill's guidance
lane_slice['is_declining_label'] = (lane_slice['trend_direction'] == 'down').astype(int)

print(f"Full dataset: {len(df)} rows -> My lane's slice: {len(lane_slice)} rows")
lane_slice[["content_id", "client_id", "impressions_90d", "content_age_days", "trend_direction", "is_declining_label"]].head()

Full dataset: 30000 rows -> My lane's slice: 30000 rows


,content_id,client_id,impressions_90d,content_age_days,trend_direction,is_declining_label
0,content_304f48230142,client_f369cb89fc,3803,187,down,1
1,content_a1fb4e703a9e,client_4e07408562,15320,445,down,1
2,content_9aa793d4d895,client_7f2253d7e2,12581,141,down,1
3,content_331d6c4de07b,client_19581e27de,11751,463,stable,0
4,content_d99b7a2d90ca,client_3fdba35f04,19140,263,down,1


### Inspecting the file system to locate `content_refresh_anonymized.csv`

In [38]:
import os

print('Listing contents of current directory:')
print(os.listdir('.'))

print('\nListing contents of data directory (if it exists):')
if os.path.exists('data'):
    print(os.listdir('data'))
    print('\nListing contents of data/raw directory (if it exists):')
    if os.path.exists('data/raw'):
        print(os.listdir('data/raw'))
    else:
        print("Directory 'data/raw' not found.")
else:
    print("Directory 'data' not found.")

Listing contents of current directory:
['requirements.txt', 'skills', 'LICENSE', 'outputs', 'work', '.github', '.git', '.gitignore', 'CLAUDE.md', 'SETUP.md', 'data', 'docs', 'README.md', 'notebooks', 'AGENTS.md', 'GUIDE.md', 'scripts', 'submission', 'DATA_USE.md']

Listing contents of data directory (if it exists):
['raw']

Listing contents of data/raw directory (if it exists):
['content_refresh_anonymized.csv']


### Cloning the starter repository

In [39]:
import os

# Clone the repository if it doesn't already exist
repo_name = 'FlyRank-internship-ml'
if not os.path.exists(repo_name):
    !git clone https://github.com/DuaaMahar5/FlyRank-internship-ml.git
    print(f"Repository '{repo_name}' cloned successfully.")
else:
    print(f"Repository '{repo_name}' already exists.")

# Change to the notebook's directory within the cloned repo for relative paths to work
# Assuming the current notebook is in work/notebooks/w02_ml_task_framing.ipynb
# We need to go up two directories from the notebook to the repo root to then access 'skills' and 'data'
# However, it's safer to just change the current working directory to the repo root for global access.

# Let's change the current working directory to the root of the cloned repository.
current_dir = os.getcwd()
repo_path = os.path.join(current_dir, repo_name)
os.chdir(repo_path)
print(f"Changed current working directory to: {os.getcwd()}")

# Now, let's list the contents of the root of the cloned repository to confirm.
print('\nListing contents of the cloned repository root:')
print(os.listdir('.'))

Cloning into 'FlyRank-internship-ml'...
remote: Enumerating objects: 134, done.
remote: Counting objects: 100% (134/134), done.
remote: Compressing objects: 100% (91/91), done.
remote: Total 134 (delta 47), reused 92 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (134/134), 1.85 MiB | 15.04 MiB/s, done.
Resolving deltas: 100% (47/47), done.
Repository 'FlyRank-internship-ml' cloned successfully.
Changed current working directory to: /content/FlyRank-internship-ml/FlyRank-internship-ml/FlyRank-internship-ml/FlyRank-internship-ml

Listing contents of the cloned repository root:
['requirements.txt', 'skills', 'LICENSE', 'outputs', 'work', '.github', '.git', '.gitignore', 'CLAUDE.md', 'SETUP.md', 'data', 'docs', 'README.md', 'notebooks', 'AGENTS.md', 'GUIDE.md', 'scripts', 'submission', 'DATA_USE.md']


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A single if-statement ignores how signals interact — content age, search volume, competition, CTR, and engagement rate all move together differently per page.

I can't yet claim this for my own model, but the starter repo's reference run shows the direction: a hand-written rule hits Precision@50 ≈ 0.24, their trained model hits ≈0.74. That ~3x gap is why I expect ML to earn its place here, pending my own test

In [40]:
import numpy as np

# Select a few key numerical features to illustrate interactions
selected_features = [
    'impressions_90d', 'content_age_days', 'search_volume', 'competition',
    'ctr', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct'
]

# Calculate descriptive statistics
display(lane_slice[selected_features].describe())

# Calculate the correlation matrix to show how signals interact
# Using .corr() which handles NaNs by default (pairwise deletion)
correlation_matrix = lane_slice[selected_features].corr()

print('\nCorrelation Matrix of Key Features:')
display(correlation_matrix.style.background_gradient(cmap='viridis', axis=None).format(precision=2))

,impressions_90d,content_age_days,search_volume,competition,ctr,engagement_rate,scroll_rate,ai_traffic_pct
count,30000.000000,30000.00000,27532.000000,27532.000000,30000.000000,30000.000000,29875.000000,30000.000000
mean,5200.366300,256.16780,158.882391,0.146954,0.510733,2.534520,18.212921,0.768196
std,16838.019547,132.70793,1518.270825,0.285241,3.279162,8.310096,29.472768,7.429454
min,1.000000,90.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,81.000000,132.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,731.000000,236.00000,10.000000,0.000000,0.070000,0.000000,5.000000,0.000000
75%,3615.250000,333.00000,20.000000,0.130000,0.290000,1.350000,23.530000,0.000000
max,517715.000000,564.00000,74000.000000,1.000000,100.000000,100.000000,300.000000,300.000000



Correlation Matrix of Key Features:


,impressions_90d,content_age_days,search_volume,competition,ctr,engagement_rate,scroll_rate,ai_traffic_pct
impressions_90d,1.00,-0.00,0.00,-0.05,-0.02,0.02,-0.10,-0.01
content_age_days,-0.00,1.00,0.09,0.05,0.01,0.04,-0.12,-0.00
search_volume,0.00,0.09,1.00,0.05,-0.00,-0.01,-0.01,-0.00
competition,-0.05,0.05,0.05,1.00,-0.02,0.00,-0.04,0.01
ctr,-0.02,0.01,-0.00,-0.02,1.00,0.10,0.01,0.01
engagement_rate,0.02,0.04,-0.01,0.00,0.10,1.00,0.16,0.03
scroll_rate,-0.10,-0.12,-0.01,-0.04,0.01,0.16,1.00,0.02
ai_traffic_pct,-0.01,-0.00,-0.00,0.01,0.01,0.03,0.02,1.00


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.